In [ ]:
!pip install 'chronos-forecasting[extras]>=2.2' scikit-learn pandas pyarrow -q


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from chronos import BaseChronosPipeline
from sklearn.linear_model import LinearRegression
from scipy.stats import f as f_dist
import warnings
warnings.filterwarnings("ignore")

PARQUET_PATH = os.environ.get("EEG_PARQUET", "/kaggle/input/datasets/michalzienkowicz/pp-eeg-4ch-raw/eeg_preprocessed_4ch_raw.parquet")
CSV_OUT = "chronos2_residual_causality_results.csv"

LIMIT_PATIENTS = None
CONTEXT_LENGTH = 512
HORIZON_LENGTH = 64
N_WINDOWS = 5
OFFSET = 1500
LAGS_TO_TEST = [5, 10, 20, 30, 40, 50]
TEST_PAIRS = [('P3', 'Fp1'), ('Fp1', 'P3'), ('P4', 'Fp2'), ('Fp2', 'P4')]

def window_starts(total_len: int, ctx: int, hor: int, n_win: int) -> np.ndarray:
    need = ctx + hor
    available_len = total_len - OFFSET
    if available_len < need: raise ValueError("Signal too short.")
    hi = total_len - need
    return np.linspace(OFFSET, hi, n_win, dtype=int)

def series_to_numpy(cell) -> np.ndarray:
    if isinstance(cell, (list, np.ndarray)): return np.asarray(cell, dtype=np.float32)
    return np.asarray(cell.tolist() if hasattr(cell, "tolist") else cell, dtype=np.float32)

REPO_ID = "amazon/chronos-2"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {REPO_ID} on {device}...")
model = BaseChronosPipeline.from_pretrained(REPO_ID, device_map=device, torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32)

def forecast_univariate(model, y_target, starts, ctx_len, hor_len):
    preds_all = []
    base_date = pd.Timestamp("2024-01-01")
    for s in starts:
        ctx_target = y_target[s : s + ctx_len]
        ctx_dates = [base_date + pd.Timedelta(seconds=x) for x in range(ctx_len)]
        df_ctx = pd.DataFrame({"item_id": ["EEG"] * ctx_len, "timestamp": ctx_dates, "target": ctx_target})
        pred_df = model.predict_df(df=df_ctx, prediction_length=hor_len, id_column="item_id", timestamp_column="timestamp", target="target", quantile_levels=[0.5])
        preds_all.append(pred_df["predictions"].values)
    return np.array(preds_all)

print("Starting Residual Causality...")
df = pd.read_parquet(PARQUET_PATH)
df_eval = df.head(LIMIT_PATIENTS).copy() if LIMIT_PATIENTS else df.copy()
results = []
done = 0

for _, row in df_eval.iterrows():
    sid = row["subject_id"]
    grp = row["group"] if "group" in row.index and pd.notna(row["group"]) else "Unknown"
    for x_col, y_col in TEST_PAIRS:
        x_signal = series_to_numpy(row[x_col])
        y_signal = series_to_numpy(row[y_col])
        starts = window_starts(len(y_signal), CONTEXT_LENGTH, HORIZON_LENGTH, N_WINDOWS)
        fm_preds = forecast_univariate(model, y_signal, starts, CONTEXT_LENGTH, HORIZON_LENGTH)
        
        for current_lag in LAGS_TO_TEST:
            pooled_residuals = []
            pooled_X_matrix = []
            for i, s in enumerate(starts):
                y_true = y_signal[s + CONTEXT_LENGTH : s + CONTEXT_LENGTH + HORIZON_LENGTH]
                res = y_true - fm_preds[i]
                pooled_residuals.extend(res)
                X_win = np.ones((HORIZON_LENGTH, 1))
                for l in range(1, current_lag + 1):
                    x_lagged = x_signal[s + CONTEXT_LENGTH - l : s + CONTEXT_LENGTH + HORIZON_LENGTH - l]
                    X_win = np.column_stack([X_win, x_lagged])
                pooled_X_matrix.append(X_win)
            
            y_res_total = np.array(pooled_residuals)
            X_full_total = np.vstack(pooled_X_matrix)
            res_mean = np.mean(y_res_total)
            rss_restricted = np.sum((y_res_total - res_mean) ** 2)
            reg_full = LinearRegression().fit(X_full_total, y_res_total)
            pred_full = reg_full.predict(X_full_total)
            rss_full = np.sum((y_res_total - pred_full) ** 2)
            obs = len(y_res_total)
            k = current_lag
            if rss_full > 0:
                f_stat = ((rss_restricted - rss_full) / k) / (rss_full / (obs - k - 1))
                p_val = f_dist.sf(f_stat, k, obs - k - 1)
            else:
                f_stat, p_val = 0.0, 1.0
            results.append({
                "subject_id": sid, "group": grp, "covariate_X": x_col, "target_Y": y_col,
                "lag_steps": current_lag, "lag_ms": int(current_lag * 2),
                "f_stat_residual": f_stat, "p_value_residual": p_val
            })
    done += 1
    if done % 10 == 0 or done == len(df_eval): print(f"Processed: {done} / {len(df_eval)}")

df_results = pd.DataFrame(results)
df_results.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT}")
